# Fine-tune Qwen to generate architectural floor layouts
Runtime → Change runtime type → pick a **GPU** (T4 is fine for the 3B model), then run the cells top to bottom.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q unsloth trl datasets

## 1. Rule engine and dataset generator
These two cells write the helper files. The engine creates and validates layouts.

In [ ]:
%%writefile layout_engine.py
"""Rule-based typical-floor layout generator + validator.
Source of truth for training data, and the safety net for model output."""
import math

TYPES = [("Studio", 35, 0), ("1-bed", 60, 1), ("2-bed", 90, 2), ("3-bed", 130, 3), ("4-bed", 1e9, 4)]
CORRIDOR = 1.6
MIN_UNIT_SIDE = 2.8


def unit_type(area):
    for name, limit, beds in TYPES:
        if area < limit:
            return name, beds


def r1(x):
    return round(x + 0.0, 1)


def generate(W, D, sf, sr, ss, cov, floors, units):
    """Plot W x D (m), front/rear/side setbacks, max coverage %, floors, total units.
    Returns a layout dict (all rects are [x, y, w, h] in plot metres, y=0 at rear) or None."""
    w, h, ox, oy = W - 2 * ss, D - sf - sr, ss, sr
    if w < 8 or h < 8:
        return None
    max_area = W * D * cov / 100
    if w * h > max_area:  # shrink footprint to respect coverage, keep it centred
        s = math.sqrt(max_area / (w * h))
        nw, nh = w * s, h * s
        ox += (w - nw) / 2
        oy += (h - nh) / 2
        w, h = nw, nh
    upf = math.ceil(units / floors)
    swap = h > w  # corridor runs along the long side
    L, B = (h, w) if swap else (w, h)
    core_l = max(3.6, min(5.0, L * 0.18))
    ud = (B - CORRIDOR) / 2  # unit depth
    seg = (L - core_l) / 2

    def M(a, b, la, lb):  # local (along, across) -> plot rect
        return [ox + b, oy + a, lb, la] if swap else [ox + a, oy + b, la, lb]

    rows = [(0, 0), (0, ud + CORRIDOR), (seg + core_l, 0), (seg + core_l, ud + CORRIDOR)]
    prio = [0, 3, 1, 2]
    out, k = [], 0
    for ri, (a0, b0) in enumerate(rows):
        c = upf // 4 + (1 if prio.index(ri) < upf % 4 else 0)
        if not c:
            continue
        uw = seg / c
        for i in range(c):
            k += 1
            area = uw * ud
            name, beds = unit_type(area)
            out.append({"id": f"A{k}", "type": name, "beds": beds, "area": r1(area),
                        "rect": [r1(v) for v in M(a0 + i * uw, b0, uw, ud)]})
    return {
        "orientation": "vertical" if swap else "horizontal",
        "footprint": [r1(v) for v in (ox, oy, w, h)],
        "units_per_floor": upf,
        "corridor": [r1(v) for v in M(0, ud, L, CORRIDOR)],
        "core": [r1(v) for v in M(seg, 0, core_l, B)],
        "units": out,
    }


def validate(inp, lay, tol=0.25):
    """Return a list of problems (empty list = layout passes)."""
    errs = []
    try:
        fx, fy, fw, fh = lay["footprint"]
        units = lay["units"]
        rects = [(u["id"], *u["rect"]) for u in units]
    except Exception:
        return ["JSON is missing footprint or units"]
    if fx < inp["ss"] - tol or fy < inp["sr"] - tol or \
       fx + fw > inp["W"] - inp["ss"] + tol or fy + fh > inp["D"] - inp["sf"] + tol:
        errs.append("footprint is outside the setbacks")
    if fw * fh > inp["W"] * inp["D"] * inp["cov"] / 100 * 1.01:
        errs.append("coverage limit exceeded")
    upf = math.ceil(inp["units"] / inp["floors"])
    if len(units) != upf:
        errs.append(f"expected {upf} units on the typical floor, got {len(units)}")
    for uid, x, y, w, h in rects:
        if x < fx - tol or y < fy - tol or x + w > fx + fw + tol or y + h > fy + fh + tol:
            errs.append(f"unit {uid} is outside the footprint")
        if min(w, h) < MIN_UNIT_SIDE:
            errs.append(f"unit {uid} is narrower than {MIN_UNIT_SIDE} m")
    for i in range(len(rects)):
        for j in range(i + 1, len(rects)):
            _, ax, ay, aw, ah = rects[i]
            _, bx, by, bw, bh = rects[j]
            if min(ax + aw, bx + bw) - max(ax, bx) > tol and min(ay + ah, by + bh) - max(ay, by) > tol:
                errs.append(f"units {rects[i][0]} and {rects[j][0]} overlap")
    return errs

In [ ]:
%%writefile make_dataset.py
"""Generate synthetic chat-format training data: inputs -> layout JSON.
Usage: python make_dataset.py --n 6000 --out data"""
import argparse
import json
import os
import random

from layout_engine import generate, validate

SYSTEM = ("You are an architectural layout engine. Given plot and program inputs, "
          "reply with only the typical-floor layout as compact JSON.")


def prompt(i):
    return (f"Plot {i['W']} x {i['D']} m. Setbacks: front {i['sf']}, rear {i['sr']}, sides {i['ss']} m. "
            f"Max coverage {i['cov']}%. Floors: {i['floors']}. Units: {i['units']}.")


def sample(rng):
    floors, upf = rng.randint(2, 12), rng.randint(2, 12)
    return {
        "W": round(rng.uniform(10, 40) * 2) / 2,
        "D": round(rng.uniform(15, 60) * 2) / 2,
        "sf": rng.choice([2, 3, 3.5, 4, 5]),
        "sr": rng.choice([1.5, 2, 3, 4]),
        "ss": rng.choice([1, 1.5, 2, 2.5]),
        "cov": rng.choice([50, 55, 60, 65, 70, 75]),
        "floors": floors,
        # sometimes uneven, but never changes units-per-floor
        "units": upf * floors - rng.randint(0, floors - 1),
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=6000)
    ap.add_argument("--out", default="data")
    ap.add_argument("--seed", type=int, default=7)
    a = ap.parse_args()
    rng, seen, rows = random.Random(a.seed), set(), []
    while len(rows) < a.n:
        inp = sample(rng)
        key = tuple(inp.values())
        if key in seen:
            continue
        lay = generate(**inp)
        if not lay or validate(inp, lay):
            continue
        avg = sum(u["area"] for u in lay["units"]) / len(lay["units"])
        if not 30 <= avg <= 160:  # keep realistic apartment sizes only
            continue
        seen.add(key)
        rows.append({"messages": [
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": prompt(inp)},
            {"role": "assistant", "content": json.dumps(lay, separators=(",", ":"))},
        ]})
    os.makedirs(a.out, exist_ok=True)
    cut = int(len(rows) * 0.95)
    for name, part in (("train", rows[:cut]), ("val", rows[cut:])):
        with open(f"{a.out}/{name}.jsonl", "w") as f:
            f.writelines(json.dumps(r) + "\n" for r in part)
    print(f"wrote {cut} train / {len(rows) - cut} val rows to {a.out}/")


if __name__ == "__main__":
    main()

In [ ]:
!python make_dataset.py --n 4000 --out data
!head -c 600 data/train.jsonl

## 2. Fine-tune (LoRA)
Takes roughly 15–40 minutes on a T4 with the 3B model.

In [ ]:
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer

# Free T4 (16 GB): 3B is comfortable. On a 24 GB+ GPU (L4/A100) use the 7B line instead.
BASE = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
# BASE = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_LEN = 2048   # layouts are short (~650 chars on average)

model, tokenizer = FastLanguageModel.from_pretrained(BASE, max_seq_length=MAX_LEN, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=32, lora_dropout=0, bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=7)

ds = load_dataset("json", data_files={"train": "data/train.jsonl", "val": "data/val.jsonl"})
ds = ds.map(lambda ex: {"text": tokenizer.apply_chat_template(ex["messages"], tokenize=False)},
            remove_columns=["messages"])

cfg = dict(dataset_text_field="text", per_device_train_batch_size=2, gradient_accumulation_steps=8,
           num_train_epochs=2, learning_rate=2e-4, lr_scheduler_type="cosine", warmup_ratio=0.03,
           logging_steps=10, eval_strategy="steps", eval_steps=100, save_strategy="no",
           bf16=is_bfloat16_supported(), fp16=not is_bfloat16_supported(),
           optim="adamw_8bit", output_dir="out/checkpoints", report_to="none")
try:    args = SFTConfig(max_seq_length=MAX_LEN, **cfg)   # older TRL
except TypeError: args = SFTConfig(max_length=MAX_LEN, **cfg)  # newer TRL
kw = dict(model=model, train_dataset=ds["train"], eval_dataset=ds["val"], args=args)
try:    trainer = SFTTrainer(tokenizer=tokenizer, **kw)
except TypeError: trainer = SFTTrainer(processing_class=tokenizer, **kw)

# Learn only from the assistant (layout JSON) tokens. Markers are for Qwen's chat template.
trainer = train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                  response_part="<|im_start|>assistant\n")
trainer.train()
model.save_pretrained("out/lora"); tokenizer.save_pretrained("out/lora")
print("saved LoRA adapter to out/lora")

## 3. Generate, validate, retry, fall back

In [ ]:
import json, re
from layout_engine import generate, validate
from make_dataset import SYSTEM, prompt

FastLanguageModel.for_inference(model)

def ask(messages, temperature):
    ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    kw = {"do_sample": True, "temperature": temperature, "top_p": 0.9} if temperature > 0 else {"do_sample": False}
    out = model.generate(input_ids=ids, max_new_tokens=2000, **kw)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def design(inp, tries=3):
    msgs = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": prompt(inp)}]
    for t in range(tries):
        text = ask(msgs, 0.0 if t == 0 else 0.4)
        try:
            lay = json.loads(re.search(r"\{.*\}", text, re.S).group(0)); errs = validate(inp, lay)
        except Exception as e:
            lay, errs = None, [f"output was not valid JSON ({e})"]
        if not errs:
            return lay, "model"
        msgs += [{"role": "assistant", "content": text},
                 {"role": "user", "content": "Fix these problems and return the full JSON again: " + "; ".join(errs[:5])}]
    return generate(**inp), "rule-engine fallback"

inp = {"W": 20, "D": 30, "sf": 3, "sr": 2, "ss": 1.5, "cov": 65, "floors": 4, "units": 16}
layout, source = design(inp)
print("source:", source)
print(json.dumps(layout, indent=1)[:1500])

In [ ]:
# Draw the layout so you can see it (works on the model's output or the fallback)
import matplotlib.pyplot as plt, matplotlib.patches as P
def draw(inp, lay):
    fig, ax = plt.subplots(figsize=(6, 8))
    ax.add_patch(P.Rectangle((0, 0), inp["W"], inp["D"], fc="#e3e9d9", ec="k"))
    ax.add_patch(P.Rectangle(tuple(lay["footprint"][:2]), *lay["footprint"][2:], fc="none", ec="b", ls="--"))
    ax.add_patch(P.Rectangle(tuple(lay["core"][:2]), *lay["core"][2:], fc="#c6cbd1", ec="k"))
    for u in lay["units"]:
        x, y, w, h = u["rect"]
        ax.add_patch(P.Rectangle((x, y), w, h, fc="#cde6d3", ec="k"))
        ax.text(x + w/2, y + h/2, f'{u["id"]}\n{u["type"]}\n{u["area"]} m²', ha="center", va="center", fontsize=7)
    ax.set_xlim(-1, inp["W"] + 1); ax.set_ylim(inp["D"] + 1, -1); ax.set_aspect("equal"); plt.show()
draw(inp, layout)

## 4. Save the adapter

In [ ]:
# Keep your trained adapter: zip and download, or copy to Google Drive
!zip -qr lora.zip out/lora
from google.colab import files; files.download("lora.zip")
# from google.colab import drive; drive.mount("/content/drive"); !cp -r out/lora /content/drive/MyDrive/arch_lora